# GeoVision-CLIP â€” Muestreo estratificado (Situacion 2)Dataset: `juanjoseorozcolopez/geovision-fuentes`GPU T4 Â· tqdm Â· checkpoint por clase Â· cache en dataset

In [ ]:
!pip install zarr tqdm -q

In [ ]:
from __future__ import annotationsimport json, time, osfrom dataclasses import dataclassfrom pathlib import Pathimport numpy as npimport pandas as pdimport xarray as xrimport torchfrom tqdm.auto import tqdmSEED = 42; TILE_PX = 64; N_BANDAS_S2 = 13; HALF = TILE_PX // 2SCL_THRESHOLD = 0.3; SCL_UMBRAL_ESCENA = 0.5; VENTANA_S2_DIAS = 5NDVI_URBANO_MAX = 0.30; RADIO_DAGMA_M = 1000; N_POR_CLASE = 1000; BATCH_SCL = 5BASE_PATH = Path("/kaggle/input/datasets/juanjoseorozcolopez/geovision-fuentes")OUT_DIR = Path("/kaggle/working"); CKPT_DIR = OUT_DIR / "checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)CLASES = ["contaminacion_alta_NO2","contaminacion_alta_SO2","ozono_anomalo","vegetacion_densa","suelo_urbano"]GUIADAS = [    ("contaminacion_alta_NO2","tropospheric_NO2_column_number_density","NO2","no2"),    ("contaminacion_alta_SO2","SO2_column_number_density","SO2","so2"),    ("ozono_anomalo","O3_column_number_density","O3","o3"),]rng = np.random.default_rng(SEED)TEXTOS = {    "contaminacion_alta_NO2": lambda t: f"Zona urbana con NO2 alto ({t.no2:.2e} mol/m2), trafico vehicular intenso.",    "contaminacion_alta_SO2": lambda t: f"Pluma industrial con SO2 elevado ({t.so2:.2e} mol/m2), corredor Yumbo-Acopi.",    "ozono_anomalo":          lambda t: f"Anomalia de ozono ({t.o3:.2e} mol/m2), fotoquimica activa.",    "vegetacion_densa":       lambda t: f"Vegetacion densa, NDVI={t.ndvi:.2f}, cana de azucar o bosque.",    "suelo_urbano":           lambda t: f"Zona urbana construida, NDVI={t.ndvi:.2f}, alta densidad edificada.",}

## 1. Abrir paneles

In [ ]:
print("Abriendo paneles...")t0 = time.time()s2    = xr.open_zarr(BASE_PATH / "copernicus_s2_sr_harmonized" / "panel.zarr", consolidated=True)no2   = xr.open_zarr(BASE_PATH / "copernicus_s5p_offl_l3_no2" / "panel.zarr", consolidated=True)so2   = xr.open_zarr(BASE_PATH / "copernicus_s5p_offl_l3_so2" / "panel.zarr", consolidated=True)o3    = xr.open_zarr(BASE_PATH / "copernicus_s5p_offl_l3_o3" / "panel.zarr", consolidated=True)era5  = xr.open_zarr(BASE_PATH / "ecmwf_era5_hourly" / "panel.zarr", consolidated=True)modis = xr.open_zarr(BASE_PATH / "modis_061_mcd19a2_granules" / "panel.zarr", consolidated=True)dagma_df = pd.read_parquet(BASE_PATH / "dagma" / "dagma_cvc_horario_raw.parquet")estaciones = pd.read_csv(BASE_PATH / "dagma" / "estaciones_metadata.csv")bands_s2 = s2["band"].values.tolist(); y_coords = s2["y"].values; x_coords = s2["x"].valuesprint(f"Paneles en {time.time()-t0:.1f}s")print(f"  S2: {dict(s2.sizes)} | NO2: {dict(no2.sizes)} | SO2: {dict(so2.sizes)} | O3: {dict(o3.sizes)}")print(f"  DAGMA: {len(dagma_df):,} filas, {len(estaciones)} estaciones")

## 2. Parsear timestamps

In [ ]:
def parse_time(s):    s = str(s)    if len(s)==8 and s[0] in "ATMP" and s[1:].isdigit():        return pd.Timestamp(year=int(s[1:5]),month=1,day=1)+pd.Timedelta(days=int(s[5:8])-1)    if "_" in s: s = s.split("_")[0]    if len(s)>=9 and s[8:9]=="T":        return pd.to_datetime(s[:15],format="%Y%m%dT%H%M%S") if len(s)!=11 else pd.to_datetime(s+"0000",format="%Y%m%dT%H%M%S")    return pd.to_datetime(s)t0 = time.time()times = {k: pd.DatetimeIndex([parse_time(t) for t in ds["time"].values])         for k,ds in [("s2",s2),("no2",no2),("so2",so2),("o3",o3),("era5",era5),("modis",modis)]}print(f"Parseados en {time.time()-t0:.1f}s")for k,v in times.items(): print(f"  {k:<6}: {v[0]} a {v[-1]} ({len(v):,})")

## 3. Pre-filtrado SCL por escena (GPU T4)

In [ ]:
SCL_CSV = "scl_por_escena.csv"out_scl = OUT_DIR / SCL_CSVinp_scl = BASE_PATH / SCL_CSVif out_scl.exists():    df_scl = pd.read_csv(out_scl)    scl_pct = df_scl["scl_pct"].values.astype("float32")    print(f"SCL cache: {out_scl.name} ({len(scl_pct)} escenas)")elif inp_scl.exists():    df_scl = pd.read_csv(inp_scl)    scl_pct = df_scl["scl_pct"].values.astype("float32")    print(f"SCL cache: {inp_scl.name} ({len(scl_pct)} escenas)")else:    assert torch.cuda.is_available(), "Activar GPU en Kaggle (Ajustes -> Acelerador -> GPU T4 x1)"    device = "cuda"    scl_da = s2["data"].sel(band="SCL")    n_t = s2.sizes["time"]; scl_pct = np.zeros(n_t, dtype="float32")    for t0 in tqdm(range(0, n_t, BATCH_SCL), desc="SCL (GPU)"):        block = scl_da.isel(time=slice(t0,t0+BATCH_SCL)).values        t = torch.from_numpy(block).to(device)        valid = ((t>=4)&(t<=7)).float(); finite = torch.isfinite(t).float()        pct = (valid.sum(dim=[-1,-2])/finite.sum(dim=[-1,-2]).clamp(min=1)).cpu().numpy()        scl_pct[t0:t0+len(pct)] = pct    pd.DataFrame({"time_idx":np.arange(n_t),"time_s2":[str(t) for t in s2["time"].values],"scl_pct":scl_pct}).to_csv(out_scl,index=False)    print(f"SCL guardado en {out_scl}")ESCENAS_LIMPIAS = np.where(scl_pct > SCL_UMBRAL_ESCENA)[0]print(f"  Escenas limpias (SCL>{SCL_UMBRAL_ESCENA}): {len(ESCENAS_LIMPIAS)}/{len(scl_pct)}")

## 4. Percentiles S5P

In [ ]:
def _perc(panel, banda, n=500):    idx = np.sort(rng.choice(panel.sizes["time"], size=min(n,panel.sizes["time"]), replace=False))    data = panel["data"].sel(band=banda).isel(time=idx).values.ravel()    data = data[np.isfinite(data)]    data = data[np.abs(data)>1e-12] if banda!="O3_column_number_density" else data[data>0]    return {p:float(np.percentile(data,p)) for p in [10,25,50,75,90,99]} | {"n":int(data.size)}perc = {"NO2":_perc(no2,"tropospheric_NO2_column_number_density"),        "SO2":_perc(so2,"SO2_column_number_density"),        "O3":_perc(o3,"O3_column_number_density")}for g,p in perc.items(): print(f"{g}: p50={p[50]:.2e} p90={p[90]:.2e} p99={p[99]:.2e} (n={p['n']:,})")

## 5. Cache check: tiles ya existen en dataset?

In [ ]:
TILES_NPZ = BASE_PATH / "tiles_train.npz"; TILES_META = BASE_PATH / "tiles_meta.parquet"CARGADOS = Falseif TILES_NPZ.exists():    print("Tiles pre-computados encontrados en dataset. Cargando...")    loaded = np.load(TILES_NPZ); tiles_arr,meta = loaded["data"],pd.read_parquet(TILES_META)    print(f"  tiles: {tiles_arr.shape} | meta: {meta.shape} filas")    print(meta['clase'].value_counts())    print(">>> Sampling saltado. Ir a celda de entrenamiento.")    CARGADOS = Trueelse:    print("No hay tiles en dataset. Ejecutando muestreo...")

## 6. Helpers

In [ ]:
@dataclassclass Tile:    t_idx:int;time_s2:str;y_idx:int;x_idx:int;lat:float;lon:float    clase:str=None;no2:float=np.nan;so2:float=np.nan;o3:float=np.nan    ndvi:float=np.nan;ndbi:float=np.nan;scl_pct:float=np.nandef extraer(t_idx, y_idx, x_idx):    return s2["data"].isel(time=t_idx, y=slice(y_idx-HALF,y_idx+HALF), x=slice(x_idx-HALF,x_idx+HALF)).valuesdef ndvi_fn(tile):    nir,red = tile[bands_s2.index("B8")].astype("float64"),tile[bands_s2.index("B4")].astype("float64")    d = np.where((nir+red)==0,np.nan,nir+red); return float(np.nanmean((nir-red)/d))def ndbi_fn(tile):    swir,nir = tile[bands_s2.index("B11")].astype("float64"),tile[bands_s2.index("B8")].astype("float64")    d = np.where((swir+nir)==0,np.nan,swir+nir); return float(np.nanmean((swir-nir)/d))def scl_tile(tile):    return float(np.isin(tile[bands_s2.index("SCL")],[4,5,6,7]).mean())PANELES = {"NO2":no2,"SO2":so2,"O3":o3}

## 7. Muestreo guiado S5P (3 clases)

In [ ]:
if not CARGADOS:    aceptados = {}    for clase, banda, key_p, key_t in GUIADAS:        ckpt_file = CKPT_DIR / f"{clase}.npz"        if ckpt_file.exists():            data = np.load(ckpt_file, allow_pickle=True)            # No se puede cargar facil, mejor re-ejecutar clases parciales es rÃ¡pido            print(f"  Checkpoint encontrado: {clase}, re-ejecutando...")        panel = PANELES[key_p]        arr = panel["data"].sel(band=banda).values        umbral = perc[key_p][99 if "O3" in key_p else 90]        t_hot, y_hot, x_hot = np.where(np.isfinite(arr)&(arr>umbral))        perm = rng.permutation(len(t_hot))        print(f"  [{clase}] {len(t_hot):,} hot pixels, umbral={umbral:.2e}")        aceptados[clase] = []; max_int = N_POR_CLASE*60        rj_ns = rj_sc = rj_bd = 0        pbar = tqdm(total=N_POR_CLASE, desc=f"  {clase}", unit="tile")        t0 = time.time()        for k in perm:            if len(aceptados[clase])>=N_POR_CLASE: break            if len(aceptados[clase])+rj_ns+rj_sc+rj_bd>=max_int: break            tp,lat_p,lon_p = int(t_hot[k]),float(panel["y"].values[int(y_hot[k])]),float(panel["x"].values[int(x_hot[k])])            dt = np.abs((times["s2"]-times[key_t][tp]).total_seconds().values)            cands = np.where(dt<VENTANA_S2_DIAS*86400)[0]            cands = np.intersect1d(cands,ESCENAS_LIMPIAS,assume_unique=True)            cands = cands[np.argsort(dt[cands])] if len(cands) else np.array([],dtype=int)            if len(cands)==0: rj_ns+=1; continue            lat = lat_p+float(rng.uniform(-0.0018,0.0018))            lon = lon_p+float(rng.uniform(-0.0018,0.0018))            y = int(np.argmin(np.abs(y_coords-lat)))            x = int(np.argmin(np.abs(x_coords-lon)))            if not(HALF<=y<s2.sizes["y"]-HALF and HALF<=x<s2.sizes["x"]-HALF): rj_bd+=1; continue            ok=False            for ts in cands[:5]:                tile=extraer(int(ts),y,x)                if tile.shape!=(N_BANDAS_S2,TILE_PX,TILE_PX): continue                if scl_tile(tile)<SCL_THRESHOLD: continue                t=Tile(t_idx=int(ts),time_s2=str(s2["time"].values[int(ts)]),y_idx=y,x_idx=x,                       lat=float(y_coords[y]),lon=float(x_coords[x]),clase=clase,                       scl_pct=scl_tile(tile),ndvi=ndvi_fn(tile),ndbi=ndbi_fn(tile))                val = float(arr[tp,int(y_hot[k]),int(x_hot[k])])                if key_p=="NO2": t.no2=val                elif key_p=="SO2": t.so2=val                else: t.o3=val                aceptados[clase].append((t,tile)); ok=True; pbar.update(1); break            if not ok: rj_sc+=1        pbar.close()        np.savez_compressed(ckpt_file, clase=clase, n=len(aceptados[clase]),                           times=[t.time_s2 for t,_ in aceptados[clase]],                           lats=[t.lat for t,_ in aceptados[clase]],                           lons=[t.lon for t,_ in aceptados[clase]])        print(f"  [{clase}] {len(aceptados[clase])}/{N_POR_CLASE} tiles en {time.time()-t0:.1f}s "              f"(rj_s2={rj_ns} rj_scl={rj_sc} rj_borde={rj_bd})")

## 8. Muestreo: vegetacion_densa

In [ ]:
if not CARGADOS:    clase = "vegetacion_densa"    aceptados[clase] = []; max_int = N_POR_CLASE*80    rj_sc = rj_cl = 0    pbar = tqdm(total=N_POR_CLASE, desc=f"  {clase}", unit="tile")    t0 = time.time()    while len(aceptados[clase])<N_POR_CLASE and len(aceptados[clase])+rj_sc+rj_cl<max_int:        t=int(rng.choice(ESCENAS_LIMPIAS)); y=int(rng.integers(HALF,s2.sizes["y"]-HALF)); x=int(rng.integers(HALF,s2.sizes["x"]-HALF))        tile=extraer(t,y,x)        if tile.shape!=(N_BANDAS_S2,TILE_PX,TILE_PX): continue        if scl_tile(tile)<SCL_THRESHOLD: rj_sc+=1; continue        nd=ndvi_fn(tile); nb=ndbi_fn(tile)        if nd>0.6:            aceptados[clase].append((Tile(t_idx=t,time_s2=str(s2["time"].values[t]),y_idx=y,x_idx=x,                lat=float(y_coords[y]),lon=float(x_coords[x]),clase=clase,scl_pct=scl_tile(tile),ndvi=nd,ndbi=nb),tile))            pbar.update(1)        else: rj_cl+=1    pbar.close()    print(f"  [{clase}] {len(aceptados[clase])}/{N_POR_CLASE} tiles en {time.time()-t0:.1f}s (rj_scl={rj_sc} rj_clase={rj_cl})")

## 9. Muestreo: suelo_urbano (guiado DAGMA)

In [ ]:
if not CARGADOS:    clase = "suelo_urbano"    RPX = int(RADIO_DAGMA_M/10)    dagma_yx = [(int(np.argmin(np.abs(y_coords-row["latitud"]))),int(np.argmin(np.abs(x_coords-row["longitud"])))) for _,row in estaciones.iterrows()]    aceptados[clase]=[]; max_int=N_POR_CLASE*40    rj_sc=rj_cl=rj_bd=0    pbar=tqdm(total=N_POR_CLASE,desc=f"  {clase}",unit="tile")    t0=time.time()    while len(aceptados[clase])<N_POR_CLASE and len(aceptados[clase])+rj_sc+rj_cl+rj_bd<max_int:        ey,ex=dagma_yx[int(rng.integers(0,len(dagma_yx)))]        y=ey+int(rng.integers(-RPX,RPX+1)); x=ex+int(rng.integers(-RPX,RPX+1))        if not(HALF<=y<s2.sizes["y"]-HALF and HALF<=x<s2.sizes["x"]-HALF): rj_bd+=1; continue        t=int(rng.choice(ESCENAS_LIMPIAS)); tile=extraer(t,y,x)        if tile.shape!=(N_BANDAS_S2,TILE_PX,TILE_PX): continue        if scl_tile(tile)<SCL_THRESHOLD: rj_sc+=1; continue        nd=ndvi_fn(tile)        if nd<NDVI_URBANO_MAX:            aceptados[clase].append((Tile(t_idx=t,time_s2=str(s2["time"].values[t]),y_idx=y,x_idx=x,                lat=float(y_coords[y]),lon=float(x_coords[x]),clase=clase,scl_pct=scl_tile(tile),ndvi=nd,ndbi=ndbi_fn(tile)),tile))            pbar.update(1)        else: rj_cl+=1    pbar.close()    print(f"  [{clase}] {len(aceptados[clase])}/{N_POR_CLASE} tiles en {time.time()-t0:.1f}s (rj_scl={rj_sc} rj_clase={rj_cl} rj_borde={rj_bd})")

## 10. Consolidar tiles

In [ ]:
if not CARGADOS:    total = sum(len(v) for v in aceptados.values())    print(f"\nTotal: {total}/{N_POR_CLASE*len(CLASES)}")    for c,lst in aceptados.items(): print(f"  {c}: {len(lst)}")    todos = [(c,t,tile) for c,lst in aceptados.items() for t,tile in lst]    tiles_arr = np.stack([tile for _,_,tile in todos])    meta = pd.DataFrame([{"clase":c,"time_s2":t.time_s2,"lat":t.lat,"lon":t.lon,        "ndvi":t.ndvi,"ndbi":t.ndbi,"scl_pct":t.scl_pct,"no2":t.no2,"so2":t.so2,"o3":t.o3,        "texto":TEXTOS[c](t)} for c,t,_ in todos])    print(f"tiles: {tiles_arr.shape} | meta: {meta.shape}")

## 11. Contexto fisico ERA5 + MODIS

In [ ]:
ERA5_BANDS = {"temperature_2m":"T2m","dewpoint_temperature_2m":"Td2m","u_component_of_wind_10m":"u10",    "v_component_of_wind_10m":"v10","boundary_layer_height":"BLH","relative_humidity_850hPa":"RH850",    "surface_pressure":"psurf","total_precipitation":"precip"}MODIS_BANDS = {"Optical_Depth_047":"AOD_047","Optical_Depth_055":"AOD_055","Column_WV":"WV"}def contexto(panel, kt, lat, lon, t_dt, bm, pref):    idx = int(np.abs((times[kt]-t_dt).total_seconds().values).argmin())    disp = panel["band"].values.tolist()    out = {}    for src,suf in bm.items():        col = f"{pref}_{suf}"        if src not in disp: out[col]=np.nan; continue        try:            v = float(panel["data"].sel(band=src).isel(time=idx).sel(y=lat,x=lon,method="nearest").values)            out[col]=v if np.isfinite(v) else np.nan        except: out[col]=np.nan    return outif not CARGADOS:    todos = [(c,t,tile) for c,lst in aceptados.items() for t,tile in lst]    t0=time.time()    ctx = [contexto(era5,"era5",t.lat,t.lon,parse_time(t.time_s2),ERA5_BANDS,"era5")|           contexto(modis,"modis",t.lat,t.lon,parse_time(t.time_s2),MODIS_BANDS,"modis")           for _,t,_ in tqdm(todos,desc="Contexto")]    df_ctx = pd.DataFrame(ctx)    meta = pd.concat([meta,df_ctx],axis=1)    print(f"Contexto: {time.time()-t0:.1f}s")    for c in df_ctx.columns: print(f"  {c:<14}: {df_ctx[c].notna().sum()}/{len(df_ctx)}")

## 12. Guardar en /kaggle/working/

In [ ]:
OUT_DIR.mkdir(parents=True,exist_ok=True)np.savez_compressed(OUT_DIR/"tiles_train.npz",data=tiles_arr,bands=np.array(bands_s2))meta.to_parquet(OUT_DIR/"tiles_meta.parquet")for f in ["tiles_train.npz","tiles_meta.parquet","scl_por_escena.csv"]:    p=OUT_DIR/f    if p.exists(): print(f"  {f}: {p.stat().st_size/1024**2:.1f} MB")

## 13. Subir nueva version del dataset (ejecutar una sola vez)

## 14. Auditoría tiles_meta.parquet — decide 0.5 vs 0.3

In [ ]:
import pandas as pd, numpy as np

meta = pd.read_parquet("/kaggle/working/tiles_meta.parquet")
meta["fecha"] = pd.to_datetime(meta["time_s2"].str[:8], format="%Y%m%d")

print("="*60); print("1. DIVERSIDAD TEMPORAL (factor clave para 0.5 vs 0.3)")
print("="*60)
for c, g in meta.groupby("clase"):
    vc = g["fecha"].value_counts()
    top5 = vc.head(5).sum()
    print(f"  {c}: {g['fecha'].nunique():>3} fechas | "
          f"max={vc.max():>3} tiles/fecha | top5={100*top5/len(g):.0f}%")

print("
"+"="*60); print("2. NDVI / NDBI POR CLASE (sanity check de pseudo-labels)")
print("="*60)
for c, g in meta.groupby("clase"):
    print(f"  {c}: NDVI={g['ndvi'].mean():+.2f}±{g['ndvi'].std():.2f}  "
          f"NDBI={g['ndbi'].mean():+.2f}±{g['ndbi'].std():.2f}  "
          f"SCL={g['scl_pct'].mean():.2f}")

print("
"+"="*60); print("3. DISTANCIA A DAGMA (suelo_urbano vs leakage LOO-CV)")
print("="*60)
est = pd.read_csv("/kaggle/input/datasets/juanjoseorozcolopez/geovision-fuentes/dagma/estaciones_metadata.csv")
urb = meta[meta["clase"]=="suelo_urbano"]
dists = urb.apply(lambda r: ((est["latitud"]-r["lat"])**2+(est["longitud"]-r["lon"])**2).pow(0.5).min()*111, axis=1)
print(f"  Distancia min a DAGMA (km): p10={dists.quantile(0.1):.2f} p50={dists.median():.2f} p90={dists.quantile(0.9):.2f} max={dists.max():.2f}")
print(f"  Tiles dentro de 1 km: {(dists<1).sum()}/{len(urb)} ({100*(dists<1).mean():.0f}%)")

print("
"+"="*60); print("4. COBERTURA ESPACIAL BBox")
print("="*60)
print(f"  lat: {meta['lat'].min():.3f} → {meta['lat'].max():.3f} (BBox: 3.30 → 3.65)")
print(f"  lon: {meta['lon'].min():.3f} → {meta['lon'].max():.3f} (BBox: -76.65 → -76.30)")
for c, g in meta.groupby("clase"):
    print(f"  {c}: lat∈[{g['lat'].min():.3f},{g['lat'].max():.3f}] lon∈[{g['lon'].min():.3f},{g['lon'].max():.3f}]")

print("
"+"="*60); print("5. CONTEXTO FÍSICO ERA5+MODIS")
print("="*60)
for col in ["era5_T2m","era5_BLH","era5_RH850","modis_AOD_055"]:
    if col in meta:
        print(f"  {col}: mean={meta[col].mean():.3g}  std={meta[col].std():.3g}  rango=[{meta[col].min():.3g},{meta[col].max():.3g}]")

print("
"+"="*60); print("VEREDICTO 0.5 vs 0.3")
print("="*60)
fmin = meta.groupby("clase")["fecha"].nunique().min()
top5_max = max(meta.groupby("clase").apply(lambda g: g["fecha"].value_counts().head(5).sum()/len(g)))
if fmin < 20 or top5_max > 0.40:
    print(f"  RE-CORRER con SCL=0.3: min_fechas={fmin}, max_top5={100*top5_max:.0f}%")
elif fmin < 40 or top5_max > 0.25:
    print(f"  MARGINAL: min_fechas={fmin}, max_top5={100*top5_max:.0f}%. 0.3 mejoraría.")
else:
    print(f"  MANTENER 0.5: min_fechas={fmin}, max_top5={100*top5_max:.0f}%.")


## 15. Subir como dataset nuevo edwardsx/geovision-tiles-sit2

In [ ]:
import json, shutil
from pathlib import Path

UP = Path("/kaggle/working/upload_tiles")
UP.mkdir(exist_ok=True)
for f in ["tiles_train.npz", "tiles_meta.parquet", "scl_por_escena.csv"]:
    src = Path("/kaggle/working") / f
    if src.exists():
        shutil.copy(src, UP / f)

(UP / "dataset-metadata.json").write_text(json.dumps({
    "title": "GeoVision Tiles Sit 2",
    "id": "edwardsx/geovision-tiles-sit2",
    "licenses": [{"name": "CC-BY-SA-4.0"}]
}, indent=2))
print(f"Listo en {UP}: {[p.name for p in UP.iterdir()]}")
!kaggle datasets create -p /kaggle/working/upload_tiles -u

## 16. Re-muestreo SOLO ozono_anomalo (SCL=0.3 + p95)

Las 4 clases restantes quedan intactas. Solo O3 se rehace con umbral mas suave para mejorar diversidad temporal.

In [ ]:
# Re-muestreo solo O3 con SCL_UMBRAL=0.3 y umbral p95
ESCENAS_LIMPIAS_03 = np.where(scl_pct > 0.3)[0]
print(f"Escenas limpias (SCL>0.3): {len(ESCENAS_LIMPIAS_03)}/{len(scl_pct)}")

# Recalcular percentil p95 de O3
_o3p = _perc(o3, "O3_column_number_density")
umbral_o3 = _o3p[75] + (_o3p[90] - _o3p[75]) * 0.5  # ~p82.5
# o usar percentil 95 directo:
n_sample = 500
idx = np.sort(rng.choice(o3.sizes['time'], size=n_sample, replace=False))
data = o3['data'].sel(band='O3_column_number_density').isel(time=idx).values.ravel()
data = data[np.isfinite(data) & (data > 0)]
umbral_o3 = float(np.percentile(data, 95))
print(f"Nuevo umbral O3 p95: {umbral_o3:.3e} (antes p99: {perc['O3'][99]:.3e})")

# Re-muestreo O3
clase = 'ozono_anomalo'
arr = o3['data'].sel(band='O3_column_number_density').values
t_hot, y_hot, x_hot = np.where(np.isfinite(arr) & (arr > umbral_o3))
print(f"  [{clase}] {len(t_hot):,} hot pixels nuevos, umbral={umbral_o3:.2e}")
perm = rng.permutation(len(t_hot))
nuevos = []
max_int = N_POR_CLASE * 60
rj_ns = rj_sc = rj_bd = 0
pbar = tqdm(total=N_POR_CLASE, desc=f'  {clase} v2', unit='tile')
t0 = time.time()
for k in perm:
    if len(nuevos) >= N_POR_CLASE: break
    if len(nuevos) + rj_ns + rj_sc + rj_bd >= max_int: break
    tp = int(t_hot[k])
    lat_p = float(o3['y'].values[int(y_hot[k])])
    lon_p = float(o3['x'].values[int(x_hot[k])])
    dt = np.abs((times['s2'] - times['o3'][tp]).total_seconds().values)
    cands = np.where(dt < VENTANA_S2_DIAS * 86400)[0]
    cands = np.intersect1d(cands, ESCENAS_LIMPIAS_03, assume_unique=True)
    cands = cands[np.argsort(dt[cands])] if len(cands) else np.array([], dtype=int)
    if len(cands) == 0: rj_ns += 1; continue
    lat = lat_p + float(rng.uniform(-0.0018, 0.0018))
    lon = lon_p + float(rng.uniform(-0.0018, 0.0018))
    y = int(np.argmin(np.abs(y_coords - lat)))
    x = int(np.argmin(np.abs(x_coords - lon)))
    if not (HALF <= y < s2.sizes['y'] - HALF and HALF <= x < s2.sizes['x'] - HALF):
        rj_bd += 1; continue
    ok = False
    for ts in cands[:5]:
        tile = extraer(int(ts), y, x)
        if tile.shape != (N_BANDAS_S2, TILE_PX, TILE_PX): continue
        if scl_tile(tile) < SCL_THRESHOLD: continue
        t = Tile(t_idx=int(ts), time_s2=str(s2['time'].values[int(ts)]),
                 y_idx=y, x_idx=x, lat=float(y_coords[y]), lon=float(x_coords[x]),
                 clase=clase, scl_pct=scl_tile(tile), ndvi=ndvi_fn(tile), ndbi=ndbi_fn(tile))
        t.o3 = float(arr[tp, int(y_hot[k]), int(x_hot[k])])
        nuevos.append((t, tile)); ok = True; pbar.update(1); break
    if not ok: rj_sc += 1
pbar.close()
print(f"  [{clase} v2] {len(nuevos)}/{N_POR_CLASE} tiles en {time.time()-t0:.1f}s (rj_s2={rj_ns} rj_scl={rj_sc} rj_borde={rj_bd})")

## 17. Reconstruir tiles_train.npz + meta con O3 nuevo

In [ ]:
# Reemplazar O3 viejo por O3 nuevo en aceptados y reconstruir todo
aceptados['ozono_anomalo'] = nuevos

todos = [(c, t, tile) for c, lst in aceptados.items() for t, tile in lst]
tiles_arr = np.stack([tile for _, _, tile in todos])
print(f"tiles: {tiles_arr.shape}")

meta = pd.DataFrame([{'clase': c, 'time_s2': t.time_s2, 'lat': t.lat, 'lon': t.lon,
    'ndvi': t.ndvi, 'ndbi': t.ndbi, 'scl_pct': t.scl_pct,
    'no2': t.no2, 'so2': t.so2, 'o3': t.o3,
    'texto': TEXTOS[c](t)} for c, t, _ in todos])

# Re-computar contexto ERA5+MODIS
ctx = [contexto(era5, 'era5', t.lat, t.lon, parse_time(t.time_s2), ERA5_BANDS, 'era5') |
       contexto(modis, 'modis', t.lat, t.lon, parse_time(t.time_s2), MODIS_BANDS, 'modis')
       for _, t, _ in tqdm(todos, desc='Contexto v2')]
meta = pd.concat([meta, pd.DataFrame(ctx)], axis=1)
print(f"meta: {meta.shape}")

# Sobrescribir outputs
np.savez_compressed(OUT_DIR / 'tiles_train.npz', data=tiles_arr, bands=np.array(bands_s2))
meta.to_parquet(OUT_DIR / 'tiles_meta.parquet')
print(f"
tiles_train.npz: {(OUT_DIR/'tiles_train.npz').stat().st_size/1024**2:.1f} MB")
print(f"tiles_meta.parquet: {(OUT_DIR/'tiles_meta.parquet').stat().st_size/1024**2:.1f} MB")

## 18. Re-auditar diversidad temporal de O3

In [ ]:
meta_v2 = pd.read_parquet('/kaggle/working/tiles_meta.parquet')
meta_v2['fecha'] = pd.to_datetime(meta_v2['time_s2'].str[:8], format='%Y%m%d')
print('Diversidad temporal v2:')
for c, g in meta_v2.groupby('clase'):
    vc = g['fecha'].value_counts()
    print(f"  {c}: {g['fecha'].nunique():>3} fechas | max={vc.max():>3} tiles/fecha | top5={100*vc.head(5).sum()/len(g):.0f}%")